# Sales Forecasting — Baseline
**Goal:** Predict daily `Revenue` and `COGS` for 2023-01-01 → 2024-07-01 using historical data (2012–2022).

**Strategy (simple seasonal average + trend):**
1. Compute average YoY growth rate from 2013–2022.
2. Build a "seasonal profile" — the average Revenue/COGS for each calendar day-of-year across all historical years.
3. Scale the profile by the projected year-level trend to produce predictions.

## 1 — Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
import xgboost as xgb
import shap

SEED = 42
np.random.seed(SEED)

DATA_DIR = 'dataset/'
TRAIN_FILE = DATA_DIR + 'sales.csv'
TEST_FILE  = DATA_DIR + 'sales_test.csv'
OUT_FILE   = DATA_DIR + 'submission.csv'

## 2 — Load & Inspect Data

In [ ]:
sales = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])
test  = pd.read_csv(TEST_FILE,  parse_dates=['Date'])

orders = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')
products = pd.read_csv(DATA_DIR + 'products.csv')
customers = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['registration_date'])
geography = pd.read_csv(DATA_DIR + 'geography.csv') 
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv', parse_dates=['date'])
promotions = pd.read_csv(DATA_DIR + 'promotions.csv', parse_dates=['start_date', 'end_date'])
inventory = pd.read_csv(DATA_DIR + 'inventory.csv', parse_dates=['date'])
shipments = pd.read_csv(DATA_DIR + 'shipments.csv', parse_dates=['ship_date', 'delivery_date'])
payments = pd.read_csv(DATA_DIR + 'payments.csv', parse_dates=['payment_date'])
returns = pd.read_csv(DATA_DIR + 'returns.csv', parse_dates=['return_date'])
reviews = pd.read_csv(DATA_DIR + 'reviews.csv', parse_dates=['review_date'])

print(f"Sales: {sales['Date'].min().date()} -> {sales['Date'].max().date()}")
print(f"Test: {test['Date'].min().date()} -> {test['Date'].max().date()}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(sales['Date'], sales['Revenue'], lw=0.7)
axes[0].set_title('Historical Daily Revenue'); axes[0].set_ylabel('Revenue')
axes[1].plot(sales['Date'], sales ['COGS'], lw=0.7, color='orange')
axes[1].set_title('Historical Daily COGS'); axes[1].set_ylabel('COGS')
plt.tight_layout()
plt.show()

## 3 — Feature Engineering

####  3.1 Daily aggregates from orders:


In [ ]:
# Kết hợp order items với product 
order_items_full = order_items.merge(
    products[['product_id', 'cost_price']], on = 'product_id', how = 'left'
)
order_items_full['item_revenue'] = order_items_full["quantity"] * order_items_full['unit_price']
order_items_full['item_cogs'] = order_items_full["quantity"] * order_items_full['cost_price']

order_agg = order_items_full.groupby('order_id').agg(
    order_revenue = ('item_revenue', 'sum'),
    order_cogs = ('item_cogs', 'sum'),
    total_items = ('quantity', 'sum'),
    distinct_products = ('product_id', 'nunique')
).reset_index()

orders_full = orders.merge(order_agg, on = 'order_id', how = 'left')

daily_orders = orders_full.groupby('order_date').agg(
    num_orders = ('order_id', 'count'),
    order_revenue = ('order_revenue', 'sum'),
    order_cogs = ('order_cogs', 'sum'),
    total_items = ('total_items', 'sum'),
    distinct_products = ('distinct_products', 'mean'),
    num_customers = ('customer_id', 'nunique')
).reset_index().rename(columns = {'order_date': 'Date'})
daily_orders['avg_order_value'] = daily_orders['order_revenue'] / daily_orders['num_orders']

#### 3.2 Features from other table: 

In [ ]:
#Web traffic
web_traffic = web_traffic.rename(columns = {'date': 'Date'})

#Aggregrate promotions
def count_active_promotions(dates, promos):
    active = np.zeros(len(dates))
    for _, row in promos.iterrows():
        mask = (dates >= row['start_date']) & (dates <= row['end_date'])
        active[mask] = 1
    return active

#Inventory
inventory_daily = inventory.groupby('date')['quantity_on_hand'].mean().reset_index()
inventory_daily.rename(columns = {'date': 'Date', 'quantity_on_hand': 'avg_inventory'}, inplace = True)

#Returns
returns_daily = returns.groupby('return_date').agg(
    num_returns = ('return_id', 'count'),
    returned_qty = ('quantity', 'sum')
).reset_index().rename(columns = {'return_date': 'Date'})

#### 3.3 Build daily dataframe

#### 3.4 Date components and cyclic features

In [ ]:
def add_date_features(df):
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['dayofyear'] = df['Date'].dt.dayofyear
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['quarter'] = df['Date'].dt.quarter
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    return df

def add_cyclical_features(df):
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    return df

In [ ]:
# --- YoY growth rate (geometric mean, 2013–2022) ---
# Use years with full data: 2013 to 2022
full_years = annual.loc[2013:2022]

yoy_rev  = full_years['Revenue'].pct_change().dropna()
yoy_cogs = full_years['COGS'].pct_change().dropna()

growth_rev  = (1 + yoy_rev).prod() ** (1 / len(yoy_rev))
growth_cogs = (1 + yoy_cogs).prod() ** (1 / len(yoy_cogs))

print(f'Geometric mean YoY Revenue growth : {growth_rev:.4f}  ({(growth_rev-1)*100:.2f}%/yr)')
print(f'Geometric mean YoY COGS    growth : {growth_cogs:.4f}  ({(growth_cogs-1)*100:.2f}%/yr)')

## 4 — Build Seasonal Profile

Average Revenue / COGS by **(month, day)** across all available years. This captures seasonal patterns while smoothing out year-specific noise.

In [ ]:
# Normalise each year so seasonal profile is scale-free
annual_means = train.groupby('year')[['Revenue','COGS']].transform('mean')
train['rev_norm']  = train['Revenue'] / annual_means['Revenue']
train['cogs_norm'] = train['COGS']    / annual_means['COGS']

# Average normalised value for each (month, day)
seasonal = (
    train
    .groupby(['month', 'day'])[['rev_norm', 'cogs_norm']]
    .mean()
    .reset_index()
)

print('Seasonal profile rows:', len(seasonal))
seasonal.head(10)

## 5 — Predict Test Period

In [ ]:
# Base level: 2022 annual mean (most recent complete year)
base_rev  = annual.loc[2022, 'Revenue']  / 365
base_cogs = annual.loc[2022, 'COGS']     / 365

# How many years ahead of 2022 is each test date?
test = test.copy()
test['month'] = test['Date'].dt.month
test['day']   = test['Date'].dt.day
test['year']  = test['Date'].dt.year
test['years_ahead'] = test['year'] - 2022

# Merge seasonal profile
test = test.merge(seasonal, on=['month', 'day'], how='left')

# Fill any missing day (e.g. Feb-29 in non-leap years) with 1.0
test['rev_norm']  = test['rev_norm'].fillna(1.0)
test['cogs_norm'] = test['cogs_norm'].fillna(1.0)

# Predicted value = base_level × growth^years_ahead × seasonal_factor
test['Revenue_pred'] = (base_rev  * growth_rev**test['years_ahead']  * test['rev_norm'] ).round(2)
test['COGS_pred']    = (base_cogs * growth_cogs**test['years_ahead'] * test['cogs_norm']).round(2)

print('Predictions sample:')
test[['Date','Revenue_pred','COGS_pred']].head(10)

## 6 — Evaluate on Training Tail (2021–2022)

Quick sanity-check: apply the same method on the last 2 years of training data and measure MAPE.

In [ ]:
val = train[train['year'].isin([2021, 2022])].copy()
val = val.merge(seasonal, on=['month', 'day'], how='left')
val['rev_norm']  = val['rev_norm'].fillna(1.0)
val['cogs_norm'] = val['cogs_norm'].fillna(1.0)
val['years_ahead'] = val['year'] - 2022  # negative for historical
val['Revenue_pred'] = base_rev  * growth_rev**val['years_ahead']  * val['rev_norm']
val['COGS_pred']    = base_cogs * growth_cogs**val['years_ahead'] * val['cogs_norm']

def mape(actual, pred):
    return (np.abs(actual - pred) / actual).mean() * 100

print(f'MAPE Revenue (2021–2022): {mape(val["Revenue"], val["Revenue_pred"]):.2f}%')
print(f'MAPE COGS    (2021–2022): {mape(val["COGS"],    val["COGS_pred"]):.2f}%')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(val['Date'], val['Revenue'],      lw=0.8, label='Actual')
ax.plot(val['Date'], val['Revenue_pred'], lw=0.8, linestyle='--', label='Predicted')
ax.set_title('Revenue — Actual vs Predicted (2021–2022 validation)')
ax.legend(); plt.tight_layout(); plt.show()

## 7 — Export Submission

In [ ]:
submission = test[['Date', 'Revenue_pred', 'COGS_pred']].rename(
    columns={'Revenue_pred': 'Revenue', 'COGS_pred': 'COGS'}
)
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission.to_csv(OUT_FILE, index=False)

print(f'Saved {len(submission)} rows to {OUT_FILE}')
submission.head(10)